> **Superseded for CoDEx-M / FB15k-237.** This notebook's full 7-variant grid only fits a 10-hour session on CoDEx-S (~28 min/run). For CoDEx-M (~220 min/run) and FB15k-237 (~304 min/run) use the 12 split notebooks (`04a_*`, `04b_*`, `04c_*`) in this folder instead, sized to run in parallel across multiple machines -- see `00_PARALLEL_RUN_PLAN.md`. This notebook is still fine to run as-is for CoDEx-S alone.

# BRGC Revision — Notebook 04: Scoring-function (selector) ablation, multi-seed (GPU)

**Purpose (Reviewer 1 c8, Reviewer 2).** The composite neighbor score is `w_d*degree + w_r*rarity - w_u*redundancy` with fixed weights (1.0, 0.35, 0.15). The single-seed CoDEx-S ablation suggested **rarity-only** matches the full score. We have decided to make **rarity-only the headline selector**; this notebook confirms that across seeds 13/42/2025 so the claim is not single-seed.

**Attach:** `brgc-frozen-source`, `brgc-kg-splits`. **GPU on.** CoDEx-S ~20 min/run (7 variants x 3 seeds = 21 runs, ~7h — fits a session or two).

Send me `scoring_ablation_<DATASET>.csv`.

In [ ]:
# === GPU + Hugging Face setup (run first) ===
import os, sys, subprocess, importlib.metadata as md, gc
from pathlib import Path
import torch
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle T4 GPU before running.")
PINNED={"transformers":"4.46.3","huggingface_hub":"0.26.5","tokenizers":"0.20.3","accelerate":"0.34.2","safetensors":"0.4.5"}
cur={}
for p in PINNED:
    try: cur[p]=md.version(p)
    except md.PackageNotFoundError: cur[p]=None
if any(cur[n]!=v for n,v in PINNED.items()):
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",*[f"{n}=={v}" for n,v in PINNED.items()]])
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
HF_CACHE_ROOT=Path("/kaggle/working/hf_cache"); HF_CACHE_ROOT.mkdir(parents=True,exist_ok=True)
HF_LOCAL_MODEL_DIR=Path("/kaggle/working/hf_models/distilbert-base-uncased"); HF_LOCAL_MODEL_DIR.parent.mkdir(parents=True,exist_ok=True)
os.environ["HF_HOME"]=str(HF_CACHE_ROOT); os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
snapshot_download(repo_id="distilbert-base-uncased", local_dir=str(HF_LOCAL_MODEL_DIR), cache_dir=str(HF_CACHE_ROOT), max_workers=1)
AutoConfig.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True)
AutoTokenizer.from_pretrained(str(HF_LOCAL_MODEL_DIR), use_fast=True, local_files_only=True)
_m=AutoModelForSequenceClassification.from_pretrained(str(HF_LOCAL_MODEL_DIR), local_files_only=True); del _m; gc.collect()
print("DistilBERT ready.")

In [ ]:
# === Stage frozen source + data (attach brgc-frozen-source and brgc-kg-splits) ===
import shutil
from pathlib import Path
INPUT_ROOT=Path("/kaggle/input"); PROJECT_DIR=Path("/kaggle/working/project"); (PROJECT_DIR/"data").mkdir(parents=True,exist_ok=True)
SRC=["config.py","data_loader.py","model.py","train.py","utils.py"]; src_dir=None
for p in INPUT_ROOT.rglob("config.py"):
    if all((p.parent/f).exists() for f in SRC): src_dir=p.parent; break
assert src_dir, "attach brgc-frozen-source"
for f in SRC: shutil.copy2(src_dir/f, PROJECT_DIR/f)
found={}
for name in ["CoDEx-S","CoDEx-M","FB15k-237"]:
    for c in INPUT_ROOT.rglob(name):
        if c.is_dir() and all((c/s).exists() for s in ["train.txt","valid.txt","test.txt"]):
            d=PROJECT_DIR/"data"/name; d.mkdir(parents=True,exist_ok=True)
            for s in ["train.txt","valid.txt","test.txt"]: shutil.copy2(c/s,d/s)
            found[name]=d; break
import subprocess,sys
assert subprocess.run([sys.executable,"-m","py_compile",*SRC],cwd=PROJECT_DIR).returncode==0
print("staged frozen_source + data:", sorted(found))

In [ ]:
# === Scoring-function (selector) ablation: config, seeds, variants ===
# Reviewer 1 comment 8 / Reviewer 2: justify the composite selector; the single-seed CoDEx-S run showed
# rarity-only is as good as the full score. This re-runs the selector across SEEDS to confirm that cleanly.
DATASET_NAME = "CoDEx-S"     # CoDEx-S is cheap (~20 min/run); extend to CoDEx-M/FB15k-237 if you have budget
SEEDS = [13, 42, 2025]
# variant -> (w_degree, w_rel_rarity, w_redundancy)
VARIANTS = {
    "full":            (1.0, 0.35, 0.15),
    "rarity_only":     (0.0, 1.00, 0.00),
    "degree_only":     (1.0, 0.00, 0.00),
    "redundancy_only": (0.0, 0.00, 1.00),
    "degree_rarity":   (1.0, 0.35, 0.00),
    "degree_redundancy":(1.0, 0.00, 0.15),
    "unscored":        (0.0, 0.00, 0.00),
}
assert DATASET_NAME in found
print(DATASET_NAME, "| seeds", SEEDS, "| variants", list(VARIANTS))

In [ ]:
# === Run selector ablation (EXPERIMENT_ID=B2, only the scoring weights change) ===
import os, sys, subprocess, json, time, shutil
from pathlib import Path
rows=[]
n=0
for vname,(wd,wr,wu) in VARIANTS.items():
    for seed in SEEDS:
        n+=1
        out_root = PROJECT_DIR/"outputs_scoring"/vname
        out_dir = out_root/DATASET_NAME/"strict_final"/"B2"/f"seed_{seed}"
        if (out_dir/"run_complete.json").exists(): print("SKIP",vname,seed); 
        else:
            if out_dir.exists(): shutil.rmtree(out_dir)
            env=os.environ.copy()
            env.update({"KGC_DATASET":DATASET_NAME,"KGC_DATA_ROOT":str(PROJECT_DIR/"data"),
                "KGC_OUTPUT_ROOT":str(out_root),"KGC_EXPERIMENT_ID":"B2","KGC_SEED":str(seed),
                "KGC_DETERMINISTIC":"1","KGC_NUM_WORKERS":"0","TOKENIZERS_PARALLELISM":"false","MASTER_PORT":str(29600+n),
                "KGC_MODEL_NAME":str(HF_LOCAL_MODEL_DIR),"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","HF_HOME":str(HF_CACHE_ROOT),
                "KGC_W_DEGREE":str(wd),"KGC_W_REL_RARITY":str(wr),"KGC_W_REDUNDANCY":str(wu)})
            t0=time.time()
            lp=out_root/f"log_{vname}_seed{seed}.log"; lp.parent.mkdir(parents=True,exist_ok=True)
            with lp.open("w") as lf:
                pr=subprocess.run([sys.executable,"train.py"],cwd=PROJECT_DIR,env=env,stdout=lf,stderr=subprocess.STDOUT,text=True)
            if pr.returncode!=0: print(lp.read_text()[-5000:]); raise RuntimeError(f"failed {vname} {seed}")
            m=[json.loads(l) for l in (out_dir/"relation_metrics.jsonl").read_text().splitlines() if l.strip()][-1]
            (out_dir/"test_metrics.json").write_text(json.dumps(m,indent=2))
            (out_dir/"run_complete.json").write_text(json.dumps({"v":vname,"seed":seed,"min":(time.time()-t0)/60}))
            print(f"DONE {vname} seed={seed} FilteredMRR={m.get('FilteredMRR'):.4f}")
        m=json.loads((out_dir/"test_metrics.json").read_text())
        rows.append([DATASET_NAME,vname,wd,wr,wu,seed,m.get("FilteredMRR"),m.get("MacroF1")])
import pandas as pd
df=pd.DataFrame(rows,columns=["Dataset","Variant","w_degree","w_rarity","w_redundancy","Seed","FilteredMRR","MacroF1"])
df.to_csv(Path("/kaggle/working")/f"scoring_ablation_{DATASET_NAME}.csv",index=False)
summ=df.groupby("Variant")[["FilteredMRR","MacroF1"]].agg(["mean","std"])
print(summ.to_string()); print("\nWrote scoring_ablation_"+DATASET_NAME+".csv  (send me this)")